In [ ]:
%%writefile test.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE  16

__global__ void matMult ( float * a, float * b, int m, int n, int k, float * c )
{
    int   bx  = blockIdx.x;
    int   by  = blockIdx.y;
    int   tx  = threadIdx.x;
    int   ty  = threadIdx.y;

    // Глобальный линейный индекс элемента вектора
    int idx = (by * BLOCK_SIZE + ty) * k + (bx * BLOCK_SIZE + tx);
    int size = m * k;  // общий размер вектора

    // Проверка границ
    if (idx < size) {
        c[idx] = a[idx] + b[idx];
    }
}

void matMultCPUSimple(float* A, float* B, float* C, int M, int N, int K) {
    int size = M * K;
    for (int i = 0; i < size; i++) {
        C[i] = A[i] + B[i];
    }
    printf("\nFirst few elements of C (CPU result):\n");
    for (int i = 0; i < (5 < size ? 5 : size); i++) {
        printf("%8.2f ", C[i]);
    }
    printf("\n");
}

int main ( int argc, char *  argv [] )
{
    int M = 1024;
    int K = 768;
    int size = M * K;
    int numBytesA = size * sizeof ( float );
    int numBytesB = size * sizeof ( float );
    int numBytesC = size * sizeof ( float );

    // allocate host memory
    float * a = new float [size];
    float * b = new float [size];
    float * c = new float [size];
    float * c_cpu = new float [size];

    // Инициализация векторов
    for ( int i = 0; i < size; i++ ) {
        a[i] = 1.0f;  // вектор из единиц
        b[i] = 2.0f;  // вектор из двоек
    }

    // CPU вычисление
    clock_t start_cpu, end_cpu;
    start_cpu = clock();
    matMultCPUSimple(a, b, c_cpu, M, N, K);
    end_cpu = clock();
    printf("CPU Simple version: %.2f ms\n", (double)(end_cpu - start_cpu) / CLOCKS_PER_SEC * 1000);

    // allocate device memory
    float * adev = NULL;
    float * bdev = NULL;
    float * cdev = NULL;

    cudaMalloc ( (void**)&adev, numBytesA );
    cudaMalloc ( (void**)&bdev, numBytesB );
    cudaMalloc ( (void**)&cdev, numBytesC );

    // set kernel launch configuration (оставляем 2D запуск для минимальных изменений)
    dim3 threads ( BLOCK_SIZE, BLOCK_SIZE );
    dim3 blocks  ( (K + BLOCK_SIZE - 1) / BLOCK_SIZE,
                   (M + BLOCK_SIZE - 1) / BLOCK_SIZE );

    // create cuda event handles
    cudaEvent_t start_gpu, stop_gpu;
    float gpuTime = 0.0f;

    cudaEventCreate ( &start_gpu );
    cudaEventCreate ( &stop_gpu );

    // asynchronously issue work to the GPU
    cudaEventRecord ( start_gpu, 0 );
    cudaMemcpy      ( adev, a, numBytesA, cudaMemcpyHostToDevice );
    cudaMemcpy      ( bdev, b, numBytesB, cudaMemcpyHostToDevice );

    matMult<<<blocks, threads>>> ( adev, bdev, M, N, K, cdev );

    cudaMemcpy      ( c, cdev, numBytesC, cudaMemcpyDeviceToHost );
    cudaEventRecord ( stop_gpu, 0 );

    cudaEventSynchronize ( stop_gpu );
    cudaEventElapsedTime ( &gpuTime, start_gpu, stop_gpu );

    // print the gpu time
    printf("\nVector size: %d elements\n", size);
    printf("GPU time: %.2f milliseconds\n", gpuTime );

    // Проверка результата
    printf("\nFirst few elements of C (GPU result):\n");
    for (int i = 0; i < (5 < size ? 5 : size); i++) {
        printf("%8.2f ", c[i]);
    }
    printf("\n");

    // Верификация результатов
    float maxDiff = 0.0f;
    for (int i = 0; i < size; i++) {
        float diff = c_cpu[i] - c[i];
        if (diff < 0) diff = -diff;
        if (diff > maxDiff) maxDiff = diff;
    }
    printf("Max difference: %e\n", maxDiff);

    // release resources
    cudaEventDestroy ( start_gpu );
    cudaEventDestroy ( stop_gpu  );
    cudaFree         ( adev  );
    cudaFree         ( bdev  );
    cudaFree         ( cdev  );

    delete[] a;
    delete[] b;
    delete[] c;
    delete[] c_cpu;

    return 0;
}

Overwriting test.cu


In [ ]:
!nvcc test.cu -o test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./test


First few elements of C (CPU result):
    3.00     3.00     3.00     3.00     3.00 
CPU Simple version: 3.65 ms

Vector size: 786432 elements
GPU time: 22.74 milliseconds

First few elements of C (GPU result):
    3.00     3.00     3.00     3.00     3.00 
Max difference: 0.000000e+00
